In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'    # Suppress TensorFlow logging (1)
import pathlib
import tensorflow as tf

tf.get_logger().setLevel('ERROR')           # Suppress TensorFlow logging (2)

# Enable GPU dynamic memory allocation
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
def download_images(folder_path):
    """This function takes the path to a folder of images as an argument,
       Returns a list of paths to each individual image"""
    image_paths = []
    image_names = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')): #if an img file
            full_path = os.path.join(folder_path, filename) #create full path to the image
            image_paths.append((full_path, filename[:-4]))
            print(filename[:-4])
    return image_paths

In [ ]:
IMAGE_PATHS = download_images("/path/to/Tensorflow/testspace/images")
IMAGE_PATHS.sort()
print(IMAGE_PATHS)

In [ ]:
import time
from object_detection.utils import label_map_util
from object_detection.utils import config_util
from object_detection.utils import visualization_utils as viz_utils
from object_detection.builders import model_builder

PATH_TO_CFG = "your_detection_model/pipeline.config" 
PATH_TO_CKPT = "your_detection_model/checkpoint"

print('Loading model... ', end='')
start_time = time.time()

# Load pipeline config and build a detection model
configs = config_util.get_configs_from_pipeline_file(PATH_TO_CFG)
model_config = configs['model']
detection_model = model_builder.build(model_config=model_config, is_training=False)

# Restore checkpoint
ckpt = tf.compat.v2.train.Checkpoint(model=detection_model)
ckpt.restore(os.path.join(PATH_TO_CKPT, 'ckpt-0')).expect_partial()

@tf.function
def detect_fn(image):
    """Detect objects in image."""

    image, shapes = detection_model.preprocess(image)
    prediction_dict = detection_model.predict(image, shapes)
    detections = detection_model.postprocess(prediction_dict, shapes)

    return detections

end_time = time.time()
elapsed_time = end_time - start_time
print('Done! Took {} seconds'.format(elapsed_time))

In [ ]:
PATH_TO_LABELS = "label_map.pbtxt"
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS,
                                                                    use_display_name=True)

In [ ]:
#Code to create an associated xml file with the image; use if wanting to generate more training data faster using a preliminary model
import numpy as np
from pathlib import Path
import xml.etree.cElementTree as ET
from PIL import Image

def create_labimg_xml(image_path, output):
    """Creates an xml file for an image based upon model output
       Input: Path to image and Tensorflow Object Detection ouput
       Output: xml file"""
    image_path = Path(image_path) #create image path
    if not os.path.exists(image_path.parent / (image_path.name[:-4]+'.xml')):#if there isn't already an xml for the img
        img = np.array(Image.open(image_path).convert('RGB')) #convert to numpy array
    
        annotation = ET.Element('annotation')
        ET.SubElement(annotation, 'folder').text = str(image_path.parent.name)
        ET.SubElement(annotation, 'filename').text = str(image_path.name)
        ET.SubElement(annotation, 'path').text = str(image_path)
    
        source = ET.SubElement(annotation, 'source')
        ET.SubElement(source, 'database').text = 'Unknown'
    
        size = ET.SubElement(annotation, 'size')
        ET.SubElement(size, 'width').text = str (img.shape[1])
        ET.SubElement(size, 'height').text = str(img.shape[0])
        ET.SubElement(size, 'depth').text = str(img.shape[2])
    
        ET.SubElement(annotation, 'segmented').text = '0'
        
        for l, x_min, y_min, x_max, y_max in output: #for each detection in detection output
            
            object = ET.SubElement(annotation, 'object')
            ET.SubElement(object, 'name').text = l
            ET.SubElement(object, 'pose').text = 'Unspecified'
            ET.SubElement(object, 'truncated').text = '0'
            ET.SubElement(object, 'difficult').text = '0'
            
            bndbox = ET.SubElement(object, 'bndbox')
            ET.SubElement(bndbox, 'xmin').text = str(x_min)
            ET.SubElement(bndbox, 'ymin').text = str(y_min)
            ET.SubElement(bndbox, 'xmax').text = str(x_max)
            ET.SubElement(bndbox, 'ymax').text = str(y_max)
            
        tree = ET.ElementTree(annotation)
        xml_file_name = image_path.parent / (image_path.name[:-4]+'.xml')
        tree.write(xml_file_name)

In [ ]:
#For running interference on a whole set of images; adapted to my ameoba detection purposes

import numpy as np
from PIL import Image
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import datetime
import warnings

def load_image_into_numpy_array(path):
    """Load an image from file into a numpy array.

    Puts image into numpy array to feed into tensorflow graph.
    Note that by convention we put it into a numpy array with shape
    (height, width, channels), where channels=3 for RGB.

    Args:
      path: the file path to the image

    Returns:
      uint8 numpy array with shape (img_height, img_width, 3)
    """
    return np.array(Image.open(path))

directory = '/path/to/all/images/' #path to images

for (root, dirs, files) in os.walk(directory):
    for dir in dirs: #for each well
        if not os.path.exists('/Users/willamrice/Desktop/Run_1_Amoebas/' + dir): #if a folder for this well's amoebas doesn't already exist
            print(f"Currently running: {dir}")
            i = 0
            IMAGE_PATHS = download_images(directory + dir)
            IMAGE_PATHS.sort()
            for image_path in IMAGE_PATHS: #current image
                print('Running inference for {}... '.format(image_path[0]), end='')
                print(image_path)
                
                image_np = load_image_into_numpy_array(image_path[0]) #put image in numpy format
                image_show = np.copy(image_np) #create copy of numpy image for show
                image_height, image_width,  _ = image_np.shape #get dimensions
                
                # Things to try:
                # Flip horizontally
                # image_np = np.fliplr(image_np).copy()
            
                # Convert image to grayscale
                # image_np = np.tile(
                #     np.mean(image_np, 2, keepdims=True), (1, 1, 3)).astype(np.uint8)
            
                input_tensor = tf.convert_to_tensor(np.expand_dims(image_np, 0), dtype=tf.float32) #convert image numpy to tensor
                detections = detect_fn(input_tensor) #get detections and probabilities
            
                # All outputs are batches tensors.
                # Convert to numpy arrays, and take index [0] to remove the batch dimension.
                # We're only interested in the first num_detections.
                num_detections = int(detections.pop('num_detections')) #get number of detections
                detections = {key: value[0, :num_detections].numpy()
                              for key, value in detections.items()}
                detections['num_detections'] = num_detections
            
                # detection_classes should be ints.
                detections['detection_classes'] = detections['detection_classes'].astype(np.int64)
            
                label_id_offset = 1
                image_np_with_detections = image_np.copy()
            
                viz_utils.visualize_boxes_and_labels_on_image_array(
                        image_np_with_detections,
                        detections['detection_boxes'],
                        detections['detection_classes']+label_id_offset,
                        detections['detection_scores'],
                        category_index,
                        use_normalized_coordinates=True,
                        max_boxes_to_draw=200,
                        min_score_thresh=.30,
                        agnostic_mode=False)
                if not os.path.exists('/Users/willamrice/Desktop/Run_1_Amoebas/' + dir + '/' + image_path[1]):
                    os.makedirs('/Users/willamrice/Desktop/Run_1_Amoebas/' + dir + '/' + image_path[1]) #make a directory if not already present
                output_directory = '/Users/willamrice/Desktop/Run_1_Amoebas/' + dir + '/' + image_path[1]
                
                # get label and coordinates of detected objects
                output = []
                threshold = 0.7 #lowest probability to be considered for detection
                offset = 0 #how much to cut out around the bounding box
                for index, score in enumerate(detections['detection_scores']):
                    if score > threshold: #if sufficient probability
                        ymin, xmin, ymax, xmax = detections['detection_boxes'][index]
                        if ymin > 0.001 and xmin > 0.001 and ymax < 0.999 and xmax < 0.999: #if not at the very edge of the image, obstructed
                            output.append(('amoeba', int(xmin * image_width), int(ymin * image_height), int(xmax * image_width), int(ymax * image_height)))
                
                # create_labimg_xml(image_path[0], output) # can use if you want to create xmls too
                
                # Save images and labels
                for l, x_min, y_min, x_max, y_max in output:
                    i += 1
                    array = np.array(image_show) #convert to array
                    image = Image.fromarray(array) 
                    cropped_img = image.crop((x_min - offset, y_min - offset, x_max + offset, y_max + offset)) #crop image from coordinates
                    #cropped_img.show()
                    file_path = output_directory + '/' + str(i) +'.jpg' #create path and title for image to save      
                    cropped_img.save(file_path, "JPEG", icc_profile=cropped_img.info.get('icc_profile'))
                # plt.figure()
                # plt.imshow(image_np_with_detections)
                print('Done')
            plt.show()

In [ ]:
#Code for performing interference on a bunch of sample images for ameobas to sort for Image Classification

import numpy as np
from PIL import Image
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import datetime
import warnings
import math

#subset
i = 1000
def load_image_into_numpy_array(path):
    """Load an image from file into a numpy array.

    Puts image into numpy array to feed into tensorflow graph.
    Note that by convention we put it into a numpy array with shape
    (height, width, channels), where channels=3 for RGB.

    Args:
      path: the file path to the image

    Returns:
      uint8 numpy array with shape (img_height, img_width, 3)
    """
    return np.array(Image.open(path))

directory = "/Users/willamrice/Downloads/Tensorflow/testspace/images" #path to sample images
output_directory = "/Users/willamrice/Downloads/Tensorflow/testspace/to_sort" #path to where to dump cropped amoeba
IMAGE_PATHS = download_images(directory)
for image_path in IMAGE_PATHS:
    print('Running inference for {}... '.format(image_path[0]), end='')

    image_np = load_image_into_numpy_array(image_path[0])  #put image in numpy format
    image_show = np.copy(image_np) #create copy of numpy image for show
    image_height, image_width,  _ = image_np.shape #get dimensions

    input_tensor = tf.convert_to_tensor(np.expand_dims(image_np, 0), dtype=tf.float32) #convert image numpy to tensor
    detections = detect_fn(input_tensor) #get detections and probabilities

    # All outputs are batches tensors.
    # Convert to numpy arrays, and take index [0] to remove the batch dimension.
    # We're only interested in the first num_detections.
    num_detections = int(detections.pop('num_detections'))
    detections = {key: value[0, :num_detections].numpy()
                  for key, value in detections.items()}
    detections['num_detections'] = num_detections

    # detection_classes should be ints.
    detections['detection_classes'] = detections['detection_classes'].astype(np.int64)

    label_id_offset = 1
    image_np_with_detections = image_np.copy()

    viz_utils.visualize_boxes_and_labels_on_image_array(
            image_np_with_detections,
            detections['detection_boxes'],
            detections['detection_classes']+label_id_offset,
            detections['detection_scores'],
            category_index,
            use_normalized_coordinates=True,
            max_boxes_to_draw=200,
            min_score_thresh=.30,
            agnostic_mode=False)
    
    # get label and coordinates of detected objects
    output = []
    threshold = 0.7 #lowest probability to be considered for detection
    for index, score in enumerate(detections['detection_scores']):
        if score > threshold: #if sufficient probability
            ymin, xmin, ymax, xmax = detections['detection_boxes'][index]
            if int(ymin*image_height) != 0 and int(xmin*image_width) != 0 and math.ceil(ymax*image_height) != image_height and math.ceil(xmax*image_width) != image_width: #if not too close to edge
                output.append(('amoeba', int(xmin * image_width), int(ymin * image_height), int(xmax * image_width), int(ymax * image_height)))
    
    
    # Save images and labels
    for l, x_min, y_min, x_max, y_max in output:
        i += 1
        array = np.array(image_show)
        image = Image.fromarray(array)
        cropped_img = image.crop((x_min, y_min, x_max, y_max)) #crop amoeba from original image based on dimensions
        #cropped_img.show()
        file_path = output_directory + '/' + str(i) +'.jpg' #create filepath for image        
        cropped_img.save(file_path, "JPEG", icc_profile=cropped_img.info.get('icc_profile')) #save amoeba into filepath
    # plt.figure()
    # plt.imshow(image_np_with_detections)
    print('Done')
# plt.show()